# Notebook 07 — Post-lock development analyses (revision)

**Purpose.** Reproduce the analyses added after review, all on the train and tune partitions only (the locked test is never read): the three-seed mechanism ladder and pretraining/width controls under the frozen 3-epoch schedule, the byte-budget sweep (uniform vs foveated at 8/16/32 local bins), the Dirichlet–multinomial vote head (CAPE-EEG v2), the patient-count learning curve, and inference latency/energy measurements. **Outputs:** `results/aggregate/table3b_*.csv`, `table6_*.csv`, `inference_measurements.json`, `paper/numbers*.tex`, `paper/figures/fig_*.pdf`. **GPU:** development pool; every run is ledgered. Runs that already exist with matching hashes are reused, never retrained.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## Three-seed ladder and controls, byte-budget sweep, Dirichlet–multinomial head (frozen 3-epoch schedule)

In [ ]:
import subprocess
for seed in ['101', '202', '303']:
    for cfgs, stage in [('B2 A1 A2 P B3 B3S B3H', None), ('B2_t16 A1_t16 B2_t8 A1_t8 P_DM', None)]:
        env = dict(os.environ, CONFIGS=cfgs, EPOCHS='3', TAG='ep3', SEED=seed, SKIP_CPU_BASELINES='1')
        p = subprocess.run(['bash', str(REPO / 'scripts' / 'run_dev_pipeline.sh')], capture_output=True, text=True, env=env)
        print('\n'.join(l for l in p.stdout.splitlines() if l.startswith('run ') or 'already PASS' in l or 'status:' in l)); assert p.returncode == 0

## Patient-count learning curve (P and B3 on 25/50/75% of training patients)

In [ ]:
for seed in ['101', '202', '303']:
    for frac in ['0.25', '0.5', '0.75']:
        env = dict(os.environ, CONFIGS='P B3', EPOCHS='3', TAG=f'lc{frac}', SEED=seed, PFRAC=frac, STAGE='learning_curve', SKIP_CPU_BASELINES='1')
        p = subprocess.run(['bash', str(REPO / 'scripts' / 'run_dev_pipeline.sh')], capture_output=True, text=True, env=env)
        print('\n'.join(l for l in p.stdout.splitlines() if 'already PASS' in l or 'status:' in l)); assert p.returncode == 0

## Inference latency and energy (idle GPU required)

In [ ]:
if not (REPO / 'results' / 'aggregate' / 'inference_measurements.json').exists():
    run(['measure_inference.py'])
im = read_json(REPO / 'results' / 'aggregate' / 'inference_measurements.json'); print('boundary:', im['boundary'][:160], '...')
for cid in ['P', 'B3']:
    r = im[cid]; print(cid, 'GPU b32 %.2f ms, %.0f win/s, %.1f W, %.3f mJ/window incremental | CPU 1 thread %.1f ms' % (r['gpu_batch_32']['median_ms'], r['gpu_batch_32']['windows_per_s'], r['gpu_batch_32']['mean_power_w'], r['gpu_batch_32']['energy_mj_per_window_incremental'], r['cpu_threads_1_batch_1']['median_ms']))

## Tables and paper figures

In [ ]:
p = subprocess.run(['bash', str(REPO / 'paper' / 'prepare_v2.sh')], capture_output=True, text=True, env=os.environ)
print('\n'.join(l for l in (p.stdout + p.stderr).splitlines() if l.strip() and not any(w in l for w in ('Warning', 'warn', 'Requested font'))))
import pandas as pd
for t in ['table3b_multiseed_ladder', 'table6_byte_sweep_paired', 'table6_learning_curve', 'table6_dirichlet_multinomial']:
    print('\n==', t); print(pd.read_csv(REPO / 'results' / 'aggregate' / f'{t}.csv').round(4).to_string(index=False))